# News RAG Aggregator

This notebook contains the complete RAG news aggregator pipeline with all components:
- **Data Retrieval**: NewsAPI and RSS feeds
- **Data Processing**: Chunking and validation
- **Embeddings**: Text embedding via OpenRouter
- **Vector Storage**: ChromaDB integration
- **Search**: Semantic, keyword, and hybrid search
- **Synthesis**: LLM-based answer generation via OpenRouter
- **Visualization**: t-SNE embeddings visualization
- **UI**: Interactive Gradio interface

**Prerequisites**: Set environment variables for `OPENROUTER_API_KEY` and `NEWSAPI_KEY`

## 1. Setup & Imports

In [ ]:
import json
import logging
import os
import re
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

import chromadb
import feedparser
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE

# Load environment variables
load_dotenv()

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

matplotlib.use("Agg")

print("✓ All imports loaded successfully")

In [ ]:
# Configuration
class Config:
    """Application configuration from environment variables."""
    OPENROUTER_API_KEY: str = os.getenv("OPENROUTER_API_KEY", "")
    OPENROUTER_MODEL: str = os.getenv("OPENROUTER_MODEL", "anthropic/claude-3.5-sonnet")
    NEWSAPI_KEY: str = os.getenv("NEWSAPI_KEY", "")
    NEWSAPI_COUNTRY: str = os.getenv("NEWSAPI_COUNTRY", "us")
    CHROMA_DB_PATH: str = "./data/chroma_db"
    RAW_ARTICLES_PATH: str = "./data/raw_articles.json"
    TSNE_PLOT_PATH: str = "./data/embeddings_tsne.png"
    MAX_QUERY_LENGTH: int = 500
    CHUNK_SIZE: int = 512
    CHUNK_OVERLAP: int = 50
    HYBRID_SEARCH_SEMANTIC_WEIGHT: float = 0.6
    HYBRID_SEARCH_KEYWORD_WEIGHT: float = 0.4
    RETRIEVAL_TOP_K: int = 5
    RSS_FEEDS: list = [
        "http://feeds.bbci.co.uk/news/rss.xml",
        "https://rss.nytimes.com/services/xml/rss/nyt/World.xml",
        "https://www.theguardian.com/world/rss",
    ]

cfg = Config()
os.makedirs("./data", exist_ok=True)
print("✓ Configuration initialized")

## 2. Security & Validation Utilities

In [ ]:
# Security & Validation
DANGEROUS_PATTERNS = [
    re.compile(r"[;\|\&\$`]"),
    re.compile(r"<script", re.IGNORECASE),
    re.compile(r"(DROP|DELETE|INSERT|UPDATE)\s", re.IGNORECASE),
]

def validate_query(query: str, min_length: int = 3, max_length: int = 500) -> tuple[bool, str]:
    """Validate a user query."""
    if not query or not query.strip():
        return False, "Query cannot be empty"
    stripped = query.strip()
    if len(stripped) < min_length:
        return False, f"Query must be at least {min_length} characters"
    if len(stripped) > max_length:
        return False, f"Query must be at most {max_length} characters"
    for pattern in DANGEROUS_PATTERNS:
        if pattern.search(stripped):
            return False, "Query contains disallowed characters"
    return True, ""

print("✓ Validation utilities loaded")

In [ ]:
# Data Fetchers
BASE_URL_NEWSAPI = "https://newsapi.org/v2/top-headlines"

def _strip_html(html: str) -> str:
    """Strip HTML tags and return plain text."""
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator=" ", strip=True)

def _fetch_full_text(url: str) -> str:
    """Attempt to scrape full article text from a URL."""
    try:
        resp = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
    except requests.RequestException:
        return ""
    
    soup = BeautifulSoup(resp.text, "html.parser")
    for tag in soup.find_all(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    
    ARTICLE_SELECTORS = ["article", '[role="main"]', ".story-body", ".article-body", ".post-content", "#article-body", ".caas-body", "main"]
    for selector in ARTICLE_SELECTORS:
        el = soup.select_one(selector)
        if el:
            text = el.get_text(separator=" ", strip=True)
            if len(text.split()) >= 50:
                return text
    
    paragraphs = soup.find_all("p")
    return " ".join(p.get_text(strip=True) for p in paragraphs)

class NewsAPIFetcher:
    """Fetches top headlines from NewsAPI.org."""
    
    def __init__(self, api_key: str, country: str = "us"):
        self.api_key = api_key
        self.country = country
    
    def fetch_top_headlines(self, category: str = "general", page_size: int = 20) -> list[dict]:
        """Fetch top headlines."""
        params = {
            "country": self.country,
            "category": category,
            "pageSize": page_size,
            "apiKey": self.api_key,
        }
        try:
            response = requests.get(BASE_URL_NEWSAPI, params=params, timeout=30)
            if response.status_code != 200:
                logger.warning(f"NewsAPI HTTP {response.status_code}")
                return []
            data = response.json()
            if data.get("status") == "error":
                logger.warning(f"NewsAPI error: {data.get('message')}")
                return []
        except Exception as e:
            logger.warning(f"NewsAPI error: {e}")
            return []
        
        articles = []
        for raw in data.get("articles", []):
            title = raw.get("title") or ""
            if not title.strip():
                continue
            content = raw.get("content") or raw.get("description") or ""
            source_obj = raw.get("source") or {}
            source = source_obj.get("name") if isinstance(source_obj, dict) else str(source_obj)
            
            articles.append({
                "title": title,
                "content": content.strip(),
                "source": source or "",
                "published_date": raw.get("publishedAt", ""),
                "url": raw.get("url", ""),
                "author": raw.get("author", ""),
            })
        
        logger.info(f"Fetched {len(articles)} articles from NewsAPI")
        return articles


class RSSFetcher:
    """Fetches and parses RSS feeds."""
    
    def __init__(self, feed_urls: Optional[list] = None):
        self.feed_urls = feed_urls or cfg.RSS_FEEDS
    
    def fetch_all(self) -> list[dict]:
        """Fetch from all configured RSS feeds."""
        seen_urls = set()
        all_articles = []
        
        for url in self.feed_urls:
            articles = self._fetch_feed(url)
            for article in articles:
                article_url = article.get("url", "")
                if article_url and article_url not in seen_urls:
                    seen_urls.add(article_url)
                    all_articles.append(article)
        
        logger.info(f"Fetched {len(all_articles)} unique articles from {len(self.feed_urls)} RSS feeds")
        return all_articles
    
    def _fetch_feed(self, url: str) -> list[dict]:
        """Fetch and parse a single RSS feed."""
        try:
            parsed = feedparser.parse(url)
        except Exception as e:
            logger.exception(f"Failed to parse feed {url}: {e}")
            return []
        
        feed_title = getattr(parsed.feed, "title", "") or ""
        articles = []
        
        for entry in parsed.entries:
            try:
                link = getattr(entry, "link", "") or ""
                title = getattr(entry, "title", "") or ""
                
                raw_content = ""
                if hasattr(entry, "content") and entry.content:
                    raw_content = getattr(entry.content[0], "value", "") or ""
                if not raw_content and hasattr(entry, "summary"):
                    raw_content = getattr(entry, "summary", "") or ""
                content = _strip_html(raw_content)
                
                if len(content.split()) < 80 and link:
                    full = _fetch_full_text(link)
                    if len(full.split()) > len(content.split()):
                        content = full
                
                articles.append({
                    "title": title,
                    "content": content,
                    "source": feed_title,
                    "published_date": getattr(entry, "published", "") or "",
                    "url": link,
                    "author": getattr(entry, "author", "") or "",
                })
            except Exception as e:
                logger.warning(f"Skipping malformed entry: {e}")
                continue
        
        logger.debug(f"Fetched {len(articles)} articles from {url}")
        return articles

    print("✓ Data fetchers loaded")

## 3. Data Processing

In [ ]:
class DataProcessor:
    """Processes and chunks news articles."""
    
    def __init__(self, chunk_size: int = 512, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
    
    def clean_text(self, raw_html: str) -> str:
        """Strip HTML tags and normalize whitespace."""
        text = BeautifulSoup(raw_html, "html.parser").get_text()
        text = re.sub(r"\s+", " ", text)
        return text.strip()
    
    def validate_article(self, article: dict) -> bool:
        """Check if an article has valid content."""
        content = article.get("content")
        if content is None or content == "":
            return False
        word_count = len(self.clean_text(content).split())
        return word_count >= 50
    
    def chunk_article(self, article: dict) -> list[dict]:
        """Split article content into overlapping chunks."""
        content = article.get("content", "")
        cleaned = self.clean_text(content)
        words = cleaned.split()
        
        url = article.get("url", "")
        source = article.get("source", "")
        published_date = article.get("published_date", "")
        article_title = article.get("title", "") or article.get("article_title", "")
        author = article.get("author", "")
        
        chunks = []
        step = self.chunk_size - self.chunk_overlap
        
        for i in range(0, len(words), step):
            chunk_words = words[i : i + self.chunk_size]
            if not chunk_words:
                break
            chunk_text = " ".join(chunk_words)
            chunk_index = len(chunks)
            chunk_id = f"{url}_{chunk_index}" if url else f"{hash(cleaned)}_{chunk_index}"
            
            chunk = {
                "text": chunk_text,
                "source": source,
                "published_date": published_date,
                "url": url,
                "article_title": article_title,
                "author": author,
                "chunk_index": chunk_index,
                "chunk_id": chunk_id,
            }
            chunks.append(chunk)
        
        return chunks
    
    def process_all(self, articles: list[dict]) -> list[dict]:
        """Validate and chunk all articles."""
        all_chunks = []
        skipped = 0
        
        for article in articles:
            if not self.validate_article(article):
                skipped += 1
                continue
            chunks = self.chunk_article(article)
            all_chunks.extend(chunks)
        
        if skipped:
            logger.info(f"Skipped {skipped} invalid articles")
        
        return all_chunks

print("✓ Data processor loaded")

In [ ]:
## 4. Embeddings & Vector Storage

In [ ]:

class Embedder:
    """Generates text embeddings via OpenRouter/OpenAI-compatible API."""
    
    def __init__(
        self,
        api_key: str,
        base_url: str = "https://openrouter.ai/api/v1",
        model: str = "openai/text-embedding-3-large",
        max_retries: int = 3,
    ):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model = model
        self.max_retries = max_retries
        self._cache = {}
    
    def embed_texts(self, texts: list[str], doc_ids: Optional[list[str]] = None) -> list[list[float]]:
        """Generate embeddings for a list of texts."""
        if doc_ids and len(doc_ids) != len(texts):
            raise ValueError("doc_ids length must match texts length")
        
        if doc_ids:
            to_embed_indices = []
            to_embed_texts = []
            results = [None] * len(texts)
            
            for i, did in enumerate(doc_ids):
                if did in self._cache:
                    results[i] = self._cache[did]
                else:
                    to_embed_indices.append(i)
                    to_embed_texts.append(texts[i])
            
            if to_embed_texts:
                new_embeddings = self._call_api(to_embed_texts)
                for idx, emb in zip(to_embed_indices, new_embeddings):
                    results[idx] = emb
                    self._cache[doc_ids[idx]] = emb
            
            return results
        
        return self._call_api(texts)
    
    def _call_api(self, texts: list[str]) -> list[list[float]]:
        """Call the embeddings API with retry."""
        for attempt in range(self.max_retries):
            try:
                response = self.client.embeddings.create(
                    input=texts,
                    model=self.model,
                )
                logger.info(f"Embedded {len(texts)} texts")
                return [item.embedding for item in response.data]
            except Exception as e:
                wait = 2**attempt
                logger.warning(f"Embedding error: {e}, retry in {wait}s")
                if attempt < self.max_retries - 1:
                    time.sleep(wait)
        
        raise RuntimeError(f"Embedding failed after {self.max_retries} retries")


class VectorStore:
    """Wraps a ChromaDB collection."""
    
    def __init__(self, persist_dir: str = "./data/chroma_db"):
        self._client = chromadb.PersistentClient(path=persist_dir)
        self._collection = None
    
    def init_collection(self, name: str = "news_articles"):
        """Create or load a named collection."""
        self._collection = self._client.get_or_create_collection(
            name=name,
            metadata={"hnsw:space": "cosine"},
        )
        logger.info(f"Collection '{name}' ready ({self._collection.count()} docs)")
    
    @property
    def collection(self):
        if self._collection is None:
            raise RuntimeError("Call init_collection() first")
        return self._collection
    
    def add_documents(
        self,
        ids: list[str],
        documents: list[str],
        embeddings: list[list[float]],
        metadatas: Optional[list[dict]] = None,
    ):
        """Upsert documents with embeddings and metadata."""
        self.collection.upsert(
            ids=ids,
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas,
        )
        logger.info(f"Upserted {len(ids)} documents")
    
    def query(
        self,
        query_embeddings: list[list[float]],
        n_results: int = 5,
        where: Optional[dict] = None,
    ) -> dict:
        """Query by embedding vectors."""
        kwargs = {
            "query_embeddings": query_embeddings,
            "n_results": n_results,
            "include": ["documents", "metadatas", "distances"],
        }
        if where:
            kwargs["where"] = where
        return self.collection.query(**kwargs)
    
    def get_all(self, include: Optional[list[str]] = None) -> dict:
        """Return all documents."""
        include = include or ["documents", "metadatas", "embeddings"]
        return self.collection.get(include=include)
    
    def get_collection_stats(self) -> dict:
        """Return stats about the collection."""
        count = self.collection.count()
        sample = {}
        if count > 0:
            peek = self.collection.peek(limit=1)
            sample = (peek.get("metadatas") or [{}])[0] if peek.get("metadatas") else {}
        return {
            "count": count,
            "sample_metadata_keys": list(sample.keys()) if sample else [],
        }
    
    def reset(self):
        """Delete all documents from the collection."""
        name = self.collection.name
        self._client.delete_collection(name)
        self._collection = self._client.get_or_create_collection(
            name=name,
            metadata={"hnsw:space": "cosine"},
        )
        logger.info(f"Collection '{name}' reset")

print("✓ Embeddings and vector storage loaded")

In [ ]:

@dataclass
class SearchResult:
    """A single search result with score and metadata."""
    text: str
    score: float
    metadata: dict[str, Any] = field(default_factory=dict)


class SemanticSearch:
    """Perform vector similarity search against ChromaDB."""
    
    def __init__(self, vector_store: VectorStore, embedder: Embedder):
        self.store = vector_store
        self.embedder = embedder
    
    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        """Embed query and retrieve most similar documents."""
        query_embedding = self.embedder.embed_texts([query])[0]
        raw = self.store.query(query_embeddings=[query_embedding], n_results=top_k)
        
        results = []
        documents = (raw.get("documents") or [[]])[0]
        distances = (raw.get("distances") or [[]])[0]
        metadatas = (raw.get("metadatas") or [[]])[0]
        
        for doc, dist, meta in zip(documents, distances, metadatas):
            score = 1.0 - dist
            results.append(SearchResult(text=doc, score=score, metadata=meta or {}))
        
        logger.debug(f"Semantic search for '{query[:50]}': {len(results)} results")
        return results


class KeywordSearch:
    """TF-IDF based keyword search."""
    
    def __init__(self):
        self._vectorizer = TfidfVectorizer(stop_words="english")
        self._tfidf_matrix = None
        self._documents = []
        self._metadatas = []
    
    def build_index(self, documents: list[str], metadatas: list[dict] = None):
        """Pre-compute the TF-IDF index."""
        self._documents = documents
        self._metadatas = metadatas or [{} for _ in documents]
        if documents:
            self._tfidf_matrix = self._vectorizer.fit_transform(documents)
        logger.info(f"Built keyword index over {len(documents)} documents")
    
    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        """Rank documents by TF-IDF similarity."""
        if self._tfidf_matrix is None or len(self._documents) == 0:
            return self._fallback_search(query, top_k)
        
        query_vec = self._vectorizer.transform([query])
        scores = (self._tfidf_matrix @ query_vec.T).toarray().flatten()
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            if scores[idx] > 0:
                results.append(
                    SearchResult(
                        text=self._documents[idx],
                        score=float(scores[idx]),
                        metadata=self._metadatas[idx],
                    )
                )
        
        logger.debug(f"Keyword search for '{query[:50]}': {len(results)} results")
        return results
    
    def _fallback_search(self, query: str, top_k: int) -> list[SearchResult]:
        """Simple substring matching."""
        query_lower = query.lower()
        results = []
        for i, doc in enumerate(self._documents):
            if query_lower in doc.lower():
                results.append(
                    SearchResult(
                        text=doc,
                        score=1.0,
                        metadata=self._metadatas[i] if i < len(self._metadatas) else {},
                    )
                )
            if len(results) >= top_k:
                break
        return results


def normalize_scores(scores: list[float]) -> list[float]:
    """Normalize scores to [0, 1] range."""
    if not scores:
        return []
    lo, hi = min(scores), max(scores)
    if hi == lo:
        return [0.0] * len(scores)
    return [(s - lo) / (hi - lo) for s in scores]


class HybridSearch:
    """Combine semantic and keyword search with weighted blending."""
    
    def __init__(self, semantic: SemanticSearch, keyword: KeywordSearch):
        self.semantic = semantic
        self.keyword = keyword
    
    def search(
        self,
        query: str,
        top_k: int = 5,
        semantic_weight: float = 0.6,
        keyword_weight: float = 0.4,
    ) -> list[SearchResult]:
        """Run both searches, blend scores, and rank."""
        fetch_k = top_k * 2
        
        sem_results = self.semantic.search(query, top_k=fetch_k)
        kw_results = self.keyword.search(query, top_k=fetch_k)
        
        sem_scores = normalize_scores([r.score for r in sem_results])
        kw_scores = normalize_scores([r.score for r in kw_results])
        
        candidates = {}
        
        for result, norm_score in zip(sem_results, sem_scores):
            key = result.metadata.get("url", result.text[:100])
            blended = norm_score * semantic_weight
            if key in candidates:
                candidates[key].score += blended
            else:
                candidates[key] = SearchResult(
                    text=result.text,
                    score=blended,
                    metadata=result.metadata,
                )
        
        for result, norm_score in zip(kw_results, kw_scores):
            key = result.metadata.get("url", result.text[:100])
            blended = norm_score * keyword_weight
            if key in candidates:
                candidates[key].score += blended
            else:
                candidates[key] = SearchResult(
                    text=result.text,
                    score=blended,
                    metadata=result.metadata,
                )
        
        ranked = sorted(candidates.values(), key=lambda r: r.score, reverse=True)
        logger.debug(f"Hybrid search: {len(ranked)} candidates -> top {top_k}")
        return ranked[:top_k]

print("✓ Search components loaded")

In [ ]:

class LLMClient:
    """OpenRouter LLM client with retry and fallback."""
    
    FALLBACK_MODELS = [
        "mistralai/mistral-7b-instruct",
        "meta-llama/llama-3-8b-instruct",
    ]
    
    def __init__(
        self,
        api_key: str,
        model: str = "anthropic/claude-3.5-sonnet",
        base_url: str = "https://openrouter.ai/api/v1",
        max_retries: int = 3,
    ):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model = model
        self.max_retries = max_retries
    
    def generate(
        self,
        system_prompt: str,
        user_prompt: str,
        temperature: float = 0.3,
        max_tokens: int = 1024,
    ) -> str:
        """Generate a chat completion with retry."""
        models_to_try = [self.model] + self.FALLBACK_MODELS
        
        for model in models_to_try:
            for attempt in range(self.max_retries):
                try:
                    response = self.client.chat.completions.create(
                        model=model,
                        messages=[
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": user_prompt},
                        ],
                        temperature=temperature,
                        max_tokens=max_tokens,
                    )
                    content = response.choices[0].message.content or ""
                    logger.info(f"LLM response from {model} ({len(content)} chars)")
                    return content
                except Exception as e:
                    wait = 2**attempt
                    logger.warning(f"LLM error on {model}: {e}, retry in {wait}s")
                    if attempt < self.max_retries - 1:
                        time.sleep(wait)
            
            logger.warning(f"All retries exhausted for model {model}, trying next")
        
        raise RuntimeError("LLM generation failed across all models")


SYSTEM_PROMPT = (
    "You are a helpful news summarization assistant. "
    "Answer the user's question using ONLY the provided articles. "
    "If information isn't in the articles, say so. "
    "Include source citations in your answer: [Source: article_title, URL]"
)


class RAGPipeline:
    """Orchestrates retrieval-augmented generation."""
    
    def __init__(
        self,
        vector_store: VectorStore,
        llm_client: LLMClient,
        config = None,
    ):
        cfg = config or cfg  # Use global cfg if not provided
        self.llm = llm_client
        self.store = vector_store
        
        embedder = Embedder(api_key=cfg.OPENROUTER_API_KEY)
        semantic = SemanticSearch(vector_store=vector_store, embedder=embedder)
        
        all_data = vector_store.get_all(include=["documents", "metadatas"])
        docs = all_data.get("documents") or []
        metas = all_data.get("metadatas") or []
        
        keyword = KeywordSearch()
        if docs:
            keyword.build_index(docs, metas)
        
        self.retriever = HybridSearch(semantic=semantic, keyword=keyword)
        self.semantic_weight = cfg.HYBRID_SEARCH_SEMANTIC_WEIGHT
        self.keyword_weight = cfg.HYBRID_SEARCH_KEYWORD_WEIGHT
        self.top_k = cfg.RETRIEVAL_TOP_K
    
    def answer_query(self, query: str) -> dict[str, Any]:
        """Validate, retrieve, synthesize, and return answer with sources."""
        is_valid, err = validate_query(query)
        if not is_valid:
            return {"answer": f"Invalid query: {err}", "sources": [], "confidence": 0.0}
        
        results = self.retriever.search(
            query,
            top_k=self.top_k,
            semantic_weight=self.semantic_weight,
            keyword_weight=self.keyword_weight,
        )
        
        if not results:
            return {
                "answer": "I couldn't find any relevant articles to answer your question.",
                "sources": [],
                "confidence": 0.0,
            }
        
        context = self._build_context(results)
        user_prompt = f"Question: {query}\n\nArticles retrieved:\n{context}"
        
        try:
            answer = self.llm.generate(
                system_prompt=SYSTEM_PROMPT,
                user_prompt=user_prompt,
            )
        except Exception as e:
            logger.error(f"LLM generation failed: {e}")
            return {
                "answer": "Sorry, I was unable to generate an answer. Please try again.",
                "sources": [],
                "confidence": 0.0,
            }
        
        sources = self._extract_sources(results)
        avg_score = sum(r.score for r in results) / len(results) if results else 0.0
        
        return {
            "answer": answer,
            "sources": sources,
            "confidence": round(avg_score, 3),
        }
    
    def _build_context(self, results: list) -> str:
        """Format retrieved results into context."""
        parts = []
        for i, r in enumerate(results, 1):
            title = r.metadata.get("article_title", "Untitled")
            source = r.metadata.get("source", "Unknown")
            url = r.metadata.get("url", "")
            parts.append(
                f"[Article {i}] {title} (Source: {source})\nURL: {url}\n{r.text}\n"
            )
        return "\n".join(parts)
    
    def _extract_sources(self, results: list) -> list[str]:
        """Extract unique source citations."""
        seen = set()
        sources = []
        for r in results:
            title = r.metadata.get("article_title", "Untitled")
            url = r.metadata.get("url", "")
            key = url or title
            if key not in seen:
                seen.add(key)
                sources.append(f"{title} - {url}" if url else title)
        return sources

print("✓ LLM client and RAG pipeline loaded")

In [ ]:

def visualize_embeddings(
    store: VectorStore,
    output_path: str = "data/embeddings_tsne.png",
    perplexity: float = 30.0,
) -> None:
    """Generate a t-SNE 2D scatter plot of embeddings."""
    data = store.get_all(include=["embeddings", "metadatas"])
    embeddings = data.get("embeddings")
    metadatas = data.get("metadatas") or []
    
    if embeddings is None or len(embeddings) < 2:
        logger.warning("Need at least 2 embeddings for t-SNE")
        return
    
    X = np.array(embeddings)
    sources = [m.get("source", "unknown") for m in metadatas]
    
    effective_perplexity = min(perplexity, len(X) - 1)
    tsne = TSNE(n_components=2, perplexity=effective_perplexity, random_state=42)
    X_2d = tsne.fit_transform(X)
    
    unique_sources = sorted(set(sources))
    color_map = {s: i for i, s in enumerate(unique_sources)}
    colors = [color_map[s] for s in sources]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(
        X_2d[:, 0], X_2d[:, 1], c=colors, cmap="tab10", alpha=0.7, s=30
    )
    
    handles = []
    for src in unique_sources:
        idx = color_map[src]
        handles.append(
            ax.scatter(
                [],
                [],
                c=[plt.cm.tab10(idx / max(len(unique_sources) - 1, 1))],
                label=src,
            )
        )
    ax.legend(handles=handles, title="Source", loc="best")
    
    ax.set_title("News Article Embeddings (t-SNE)")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info(f"t-SNE plot saved to {output_path} ({len(X)} points)")
    print(f"✓ Visualization saved to {output_path}")

print("✓ Visualization function loaded")

In [ ]:
from IPython.display import Image, display

# generate the t‑SNE plot (will write to cfg.TSNE_PLOT_PATH)
visualize_embeddings(store, output_path=cfg.TSNE_PLOT_PATH)

# show the resulting image inline
display(Image(cfg.TSNE_PLOT_PATH))

In [ ]:

def fetch_articles() -> list[dict]:
    """Fetch articles from NewsAPI and RSS feeds."""
    articles = []
    
    print("🔍 Fetching from NewsAPI...")
    try:
        if cfg.NEWSAPI_KEY:
            fetcher = NewsAPIFetcher(api_key=cfg.NEWSAPI_KEY, country=cfg.NEWSAPI_COUNTRY)
            fetched = fetcher.fetch_top_headlines(page_size=30)
            articles.extend(fetched)
            print(f"  ✓ Got {len(fetched)} articles from NewsAPI")
        else:
            print("  ⚠ NEWSAPI_KEY not set, skipping NewsAPI")
    except Exception as e:
        print(f"  ✗ NewsAPI fetch failed: {e}")
    
    print("🔍 Fetching from RSS feeds...")
    try:
        rss = RSSFetcher(feed_urls=cfg.RSS_FEEDS)
        rss_articles = rss.fetch_all()
        articles.extend(rss_articles)
        print(f"  ✓ Got {len(rss_articles)} articles from RSS")
    except Exception as e:
        print(f"  ✗ RSS fetch failed: {e}")
    
    processor = DataProcessor(
        chunk_size=cfg.CHUNK_SIZE, chunk_overlap=cfg.CHUNK_OVERLAP
    )
    chunks = processor.process_all(articles)
    print(f"✓ Processed {len(articles)} articles into {len(chunks)} chunks")
    
    os.makedirs("./data", exist_ok=True)
    with open(cfg.RAW_ARTICLES_PATH, "w") as f:
        json.dump(chunks, f, indent=2, default=str)
    print(f"✓ Saved to {cfg.RAW_ARTICLES_PATH}")
    
    return chunks


def embed_articles() -> int:
    """Generate embeddings and store in ChromaDB."""
    if not os.path.exists(cfg.RAW_ARTICLES_PATH):
        print("⚠ No cached articles. Run fetch_articles() first.")
        return 0
    
    with open(cfg.RAW_ARTICLES_PATH) as f:
        chunks = json.load(f)
    
    if not chunks:
        print("⚠ No chunks to embed.")
        return 0
    
    print(f"🤖 Embedding {len(chunks)} chunks...")
    embedder = Embedder(api_key=cfg.OPENROUTER_API_KEY)
    texts = [c["text"] for c in chunks]
    embeddings = embedder.embed_texts(texts)
    
    store = VectorStore(persist_dir=cfg.CHROMA_DB_PATH)
    store.init_collection("news_articles")
    
    ids = [c.get("chunk_id", str(i)) for i, c in enumerate(chunks)]
    metadatas = [
        {
            "source": c.get("source", ""),
            "published_date": c.get("published_date", ""),
            "url": c.get("url", ""),
            "chunk_index": c.get("chunk_index", 0),
            "article_title": c.get("article_title", ""),
            "author": c.get("author", ""),
        }
        for c in chunks
    ]
    
    store.add_documents(
        ids=ids, documents=texts, embeddings=embeddings, metadatas=metadatas
    )
    
    stats = store.get_collection_stats()
    print(f"✓ Stored {stats['count']} chunks in ChromaDB")
    
    return stats['count']


# Initialize the RAG pipeline globally
print("⏳ Initializing RAG pipeline...")
store = VectorStore(persist_dir=cfg.CHROMA_DB_PATH)
store.init_collection("news_articles")
llm = LLMClient(api_key=cfg.OPENROUTER_API_KEY, model=cfg.OPENROUTER_MODEL)
pipeline = RAGPipeline(vector_store=store, llm_client=llm, config=cfg)

stats = store.get_collection_stats()
print(f"✓ Pipeline ready. Articles in index: {stats['count']}")

if stats['count'] == 0:
    print("\n⚠ No articles indexed yet. Run the cells below to fetch and embed articles.")

In [ ]:

def query_news(question: str) -> tuple[str, str, str]:
    """Run RAG query and return (answer, sources_text, confidence)."""
    if not question or not question.strip():
        return "Please enter a question.", "", "No confidence data"
    
    result = pipeline.answer_query(question.strip())
    sources_text = "\n".join(f"• {s}" for s in result["sources"]) if result["sources"] else "No sources found."
    confidence_str = f"Confidence: {result['confidence']:.1%}"
    
    return result["answer"], sources_text, confidence_str


# Import Gradio
import gradio as gr

# Create the Gradio interface
def run_ui():
    """Launch the Gradio UI."""
    with gr.Blocks(title="News RAG Aggregator", theme=gr.themes.Soft()) as demo:
        gr.Markdown("""
# 📰 News RAG Aggregator
Ask questions about recent news. Answers are synthesized from indexed articles with citations.
        """)
        
        with gr.Row():
            with gr.Column():
                question = gr.Textbox(
                    label="Your question",
                    placeholder="e.g. What are the latest developments in AI?",
                    lines=3,
                )
                submit_btn = gr.Button("🔍 Search", variant="primary")
        
        with gr.Row():
            answer_out = gr.Textbox(label="📝 Answer", lines=10, interactive=False)
        
        with gr.Row():
            with gr.Column():
                sources_out = gr.Textbox(label="📚 Sources", lines=5, interactive=False)
            with gr.Column():
                confidence_out = gr.Textbox(label="📊 Confidence", lines=1, interactive=False)
        
        submit_btn.click(
            fn=query_news,
            inputs=question,
            outputs=[answer_out, sources_out, confidence_out]
        )
        
        # Add example questions
        gr.Examples(
            examples=[
                ["What are the latest developments in artificial intelligence?"],
                ["What happened in international politics recently?"],
                ["What are the major business news?"],
            ],
            inputs=question,
        )
    
    return demo

# Launch the UI
print("🚀 Launching Gradio interface...\n")
demo = run_ui()
demo.launch(share=False)

🚀 Launching Gradio interface...



/var/folders/02/7h349ywn7yx3vfkg1pvs4rv40000gn/T/ipykernel_44031/2905875838.py:19: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="News RAG Aggregator", theme=gr.themes.Soft()) as demo:
2026-03-05 10:17:28,538 - httpx - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-05 10:17:28,907 - httpx - INFO - HTTP Request: GET http://127.0.0.1:7863/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-05 10:17:28,933 - httpx - INFO - HTTP Request: HEAD http://127.0.0.1:7863/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


2026-03-05 10:17:43,411 - httpx - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-03-05 10:17:43,672 - __main__ - INFO - Embedded 1 texts
2026-03-05 10:17:44,251 - httpx - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-05 10:17:58,792 - __main__ - INFO - LLM response from openai/gpt-5-nano (815 chars)
